# Handelsblatt Konjunkturtracker

### 0.Setup
0.1 Load Programs

In [289]:
#Programme laden
import pandas as pd
import io 
import numpy as np
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials
import os
import json 
import pygsheets
from datawrapper import Datawrapper
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pyreadr
import tempfile
import gzip


0.2 Load Data

In [290]:
# Google Sheet verbinden
SERVICE_ACCOUNT_FILE_1 = '/Users/bb/Desktop/handelsblatt/Konjunkturprognosetracker/konjunkturtracker-f3c2912f6ee8.json'
SHEET_ID = "1Sd9x9xJDM9i8Pa-CjKNuXMGUfYTAhnQFaHrmzK-ZOSg" 
gc = pygsheets.authorize(service_file=SERVICE_ACCOUNT_FILE_1)
sh = gc.open_by_key(SHEET_ID)

# Daten aus dem Google Sheet laden
old_data = {}

# UNEMP
ws1 = sh[0] 
old_data["unemp"] = pd.DataFrame(ws1.get_all_records())

# CPI
ws2 = sh[1] 
old_data["cpi"]  = pd.DataFrame(ws2.get_all_records())

# GDP
ws3 = sh[2] 
old_data["gdp"]  = pd.DataFrame(ws3.get_all_records())


old_data


{'unemp':          Vintage   Variable                 Institute  Year Value  \
 0     2022-06-21   UNEMPRBA                       IWH  2022     5   
 1     2022-09-08   UNEMPRBA                       IWH  2022   5,3   
 2     2022-12-20   UNEMPRBA                       IWH  2022   5,3   
 3     2023-03-15   UNEMPRBA                       IWH  2023   5,4   
 4     2023-06-22   UNEMPRBA                       IWH  2023   5,6   
 ...          ...        ...                       ...   ...   ...   
 4233  2022-01-14   UNEMPRBA  observed (first release)  2021   5,7   
 4234  2023-01-13   UNEMPRBA  observed (first release)  2022   5,3   
 4235  2024-01-15   UNEMPRBA  observed (first release)  2023   5,7   
 4236  2025-01-15   UNEMPRBA  observed (first release)  2024     6   
 4237  2026-01-15   UNEMPRBA  observed (first release)  2025   6,3   
 
                                        Name  
 0     Halle Institute for Economic Research  
 1     Halle Institute for Economic Research  
 2     H

0.3 Update Data

In [291]:
# Mit dem Google Drive verbinden
SERVICE_ACCOUNT_FILE_2 = "/Users/bb/Desktop/handelsblatt/Konjunkturprognosetracker/konjunktur-ca3c010ecde9.json"
SCOPES = ["https://www.googleapis.com/auth/drive"]
FOLDER_ID = "1NzvJLZlWKCxfo7ujFZ7SZk02xD44EteM"
c = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE_2,scopes=SCOPES)
gd = build("drive", "v3", credentials=c)

# Dokumente aus dem Google Drive laden
q = f"'{FOLDER_ID}' in parents"
results = gd.files().list(q=q, fields="files(id, name, mimeType)").execute()
files = results.get("files", [])

# Namen aus den Dateien checken
for file in files: print(f'{file["name"]}')



data_handelsblatt_2026.rds
data_handelsblatt_old.rds


In [292]:
new_data = []

for file in files:
    id = file["id"]
    name = file["name"]
    type = file["mimeType"]

    # Download
    r = gd.files().get_media(fileId=id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, r)

    # Loop 
    done = False
    while not done:
        status, done = downloader.next_chunk()

    fh.seek(0)
    zip = fh.read()

    # Datei auspacken
    rds = gzip.decompress(zip)

    # RDS-Datei temporär speichern
    with tempfile.NamedTemporaryFile(suffix=".rds") as tmp:
        tmp.write(rds)
        tmp.flush()
        result = pyreadr.read_r(tmp.name)

    # In ein Datenframe speichern 
    df = list(result.values())[0]
    df["source"] = name
    new_data.append(df)

new_data

[    Variable Institute     Vintage  Quarter  Month  Day    Year  Value  \
 0   UNEMPRBA      BMWE  2026-01-28        1    1.0   28  2026.0    6.2   
 1   UNEMPRBA      BMWE  2026-01-28        1    1.0   28  2027.0    6.0   
 2        GDP      BMWE  2026-01-28        1    1.0   28  2026.0    1.0   
 3        GDP      BMWE  2026-01-28        1    1.0   28  2027.0    1.3   
 4        GDP      BMWE  2026-01-28        1    1.0   28  2028.0    0.9   
 5        CPI      BMWE  2026-01-28        1    1.0   28  2026.0    2.1   
 6        CPI      BMWE  2026-01-28        1    1.0   28  2027.0    2.0   
 7        GDP    Helaba  2026-01-27        1    1.0   27  2026.0    1.7   
 8        GDP    Helaba  2026-01-27        1    1.0   27  2027.0    1.5   
 9        CPI    Helaba  2026-01-27        1    1.0   27  2026.0    2.7   
 10       CPI    Helaba  2026-01-27        1    1.0   27  2027.0    2.5   
 11       GDP       IMF  2026-01-19        1    1.0   19  2026.0    1.1   
 12       GDP       IMF  

In [293]:
# Datensätze zusammenführen
all_data = {}

for df in new_data:
    df = df[["Variable", "Institute", "Vintage","Year", "Value"]]

    cpi_new = df[df["Variable"] == "CPI"].copy()
    all_data["cpi"] = pd.concat([old_data["cpi"], cpi_new], ignore_index=True)

    gdp_new = df[df["Variable"] == "GDP"].copy()
    all_data["gdp"] = pd.concat([old_data["gdp"], gdp_new], ignore_index=True)

    unemp_new = df[df["Variable"] == "UNEMPRBA"].copy()
    all_data["unemp"] = pd.concat([old_data["unemp"], unemp_new], ignore_index=True)


all_data


{'cpi':          Vintage Variable    Institute    Year Value
 0     2000-01-15      CPI       Helaba  2000.0   1,4
 1     2000-02-15      CPI       Helaba  2000.0   1,5
 2     2000-03-15      CPI       Helaba  2000.0   1,6
 3     2000-04-15      CPI       Helaba  2000.0   1,6
 4     2000-05-15      CPI       Helaba  2000.0   1,5
 ...          ...      ...          ...     ...   ...
 5851  2015-01-15      CPI       Helaba  2016.0   1.7
 5852  2015-01-15      CPI         Deka  2015.0   0.7
 5853  2015-01-15      CPI         Deka  2016.0   1.6
 5854  2015-01-15      CPI  DB Research  2015.0   1.0
 5855  2015-01-15      CPI  DB Research  2016.0   1.5
 
 [5856 rows x 5 columns],
 'gdp':          Vintage Variable    Institute    Year Value
 0       15.06.22      GDP          RWI  2022.0   1,9
 1       08.09.22      GDP          RWI  2022.0   1,1
 2       15.12.22      GDP          RWI  2022.0   1,8
 3       16.03.23      GDP          RWI  2023.0   0,2
 4       15.06.23      GDP          RWI 

0.4 Prepare Data

In [316]:
#Beobachtete Werte und Prognosen trennen; dann wieder mergen

observed = {}
prog = {}
merged = {}

for key, df in all_data.items():
    # Daten reinigen
    df["Year"]    = pd.to_numeric(df["Year"], errors="coerce")
    df["Vintage"] = pd.to_datetime(df["Vintage"], dayfirst=True, errors="coerce")
    df["Value"] = df["Value"].astype(str).str.replace(",", ".", regex=False)
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    # Institute umbenennen
    df["Institute"] = df["Institute"].replace({"Kiel Institute": "IfW Kiel"})
    df["Institute"] = df["Institute"].replace({"DIW": "DIW Berlin"})
    df["Institute"] = df["Institute"].replace({"IWH": "IWH Halle"})
    df["Institute"] = df["Institute"].replace({"ifo": "ifo München"})
    df["Institute"] = df["Institute"].replace({"RWI": "RWI Essen"})
    df["Institute"] = df["Institute"].replace({"HRI": "<b>HRI</b>"})
    #df["Institute"] = df["Institute"].replace({"SVR": "Wirtschaftsweisen"})
    #df["Institute"] = df["Institute"].replace({"BMWE": "Bundesregierung"})
    #df["Institute"] = df["Institute"].replace({"EC": "EU Kommission"})

    # Eingetroffene Werte filtern und kennzeichen 
    observed[key] = df[df["Institute"] == "observed (first release)"][["Year", "Value"]]
    observed[key] = observed[key].rename(columns={"Value": "Observed_Value"})

    # Prognosedatensatz filtern 
    prog[key] = df[~df["Institute"].str.startswith("observed", na=False)]
    prog[key] = prog[key][["Variable","Vintage","Year", "Institute", "Value"]]


# Loop durch den Merge 
for key in ["cpi", "gdp", "unemp"]: merged[key] = prog[key].merge(observed[key],on="Year",how="left")

merged



{'cpi':       Variable    Vintage    Year    Institute  Value  Observed_Value
 0          CPI 2000-01-15  2000.0       Helaba    1.4             0.6
 1          CPI 2000-01-15  2000.0       Helaba    1.4             0.6
 2          CPI 2000-01-15  2000.0       Helaba    1.4             0.6
 3          CPI 2000-01-15  2000.0       Helaba    1.4             0.6
 4          CPI 2000-01-15  2000.0       Helaba    1.4             0.6
 ...        ...        ...     ...          ...    ...             ...
 46725      CPI 2015-01-15  2016.0  DB Research    1.5             1.7
 46726      CPI 2015-01-15  2016.0  DB Research    1.5             1.7
 46727      CPI 2015-01-15  2016.0  DB Research    1.5             1.7
 46728      CPI 2015-01-15  2016.0  DB Research    1.5             1.7
 46729      CPI 2015-01-15  2016.0  DB Research    1.5             1.7
 
 [46730 rows x 6 columns],
 'gdp':       Variable    Vintage    Year    Institute  Value  Observed_Value
 0          GDP 2022-06-15  2022.0

0.5 Split Data

In [317]:
# Nur die letzten Prognosen benutzten
# Nur die letzten und aktuellsten Prognosen benutzen

aktuell = {}
aktuell_t1 = {}

for key in ["cpi", "unemp", "gdp"]:
    df = merged[key].copy()

    # Nur die "letzte" Prognose pro Jahr benutzen 
    akt = df.groupby(["Institute", "Year"])["Vintage"].idxmax()
    df_latest = df.loc[akt].reset_index(drop=True)
    
    # Das Jahr dynamisch definieren 
    t2 = df["Year"].max()
    t1 = t2 -1
    df_latest_t1 = df_latest[df_latest["Year"] ==  t1]

    # Jahr Variable als String ohne Komma umwandeln
    df_latest["Year"] = df_latest["Year"].astype(float).astype(int).astype(str)
    df_latest_t1["Year"] = df_latest_t1["Year"].astype(float).astype(int).astype(str)

    # Daten auf die Dictionaries spielen
    aktuell[key] = df_latest
    aktuell_t1[key] = df_latest_t1[["Variable", "Vintage", "Year", "Institute", "Value"]]


aktuell_t1["unemp"]


,Variable,Vintage,Year,Institute,Value
11,UNEMPRBA,2026-01-02,2026,<b>HRI</b>,6.5
32,UNEMPRBA,2025-12-19,2026,BBK,6.2
60,UNEMPRBA,2026-01-28,2026,BMWE,6.2
88,UNEMPRBA,2025-12-18,2026,DB Research,3.1
116,UNEMPRBA,2025-12-12,2026,DIW Berlin,6.2
142,UNEMPRBA,2026-01-15,2026,Deka,6.3
170,UNEMPRBA,2025-09-23,2026,GD,6.1
198,UNEMPRBA,2025-12-04,2026,HWWI/HWWA,5.9
226,UNEMPRBA,2025-12-18,2026,Helaba,6.1
254,UNEMPRBA,2025-09-24,2026,IAB,6.3


### Grafik 1 - Die aktuellsten Prognosen im Median

In [318]:
#Median berechenen und als Zeile einfügen 

top5_institute = ["Median", "DIW Berlin", "IfW Kiel", "RWI Essen", "IWH Halle", "ifo München"]
#top6_institute = ["Median", "DIW Berlin", "IfW Kiel", "RWI Essen", "IWH Halle", "ifo München", "HRI"]
top6_institute = ["Median", "DIW Berlin", "IfW Kiel", "RWI Essen", "IWH Halle", "ifo München", "<b>HRI</b>"]
#top5_institute = ["Median", "DIW Berlin", "ifo München", "Wirtschaftsweisen", "Bundesregierung", "EU Kommission"]
grafik_1 = {}
for key in ["unemp", "cpi","gdp"]:
    df = aktuell_t1[key].copy()

    median = df["Value"].median()
    #median_top5 = df[df["Institute"].isin(top5_institute)]["Value"].median()

    median_row = {"Institute": "Alle", "Year": df["Year"].iloc[0], "Prognose" : median }
    #median_top5_row = {"Institute": "Top 5 Institute", "Year": df["Year"].iloc[0], "Prognose" : median_top5 }

    df = df[df["Institute"].isin(top6_institute)].reset_index(drop=True)
    df = pd.concat([df, pd.DataFrame([median_row])], ignore_index=True)
    #df = pd.concat([df, pd.DataFrame([median_row]), pd.DataFrame([median_top5_row])], ignore_index=True)
    df = df[["Variable", "Year", "Institute", "Value", "Prognose"]]
    
    # Einteilung erstellen
    df["einteilung"] = "Das sagen die einzelnden Institute"
    df.loc[df["Institute"] == "Alle", "einteilung"] = "Alle Prognosen im Median"
    #df.loc[df["Institute"] == "HRI", "einteilung"] = "Das sagt das Handelsblatt Research Institute"
    #df.loc[df["Institute"] == "Medianprognose der Top 5 Institute", "einteilung"] = "Medianprognose der Top 5 Institute"
    #df.loc[df["Institute"] == "Top 5 Institute", "einteilung"] = "Das zeigt die Medianprognose"

    # Reihenfolge ändern
    df = pd.concat([df[df["Institute"] == "Alle"], df[df["Institute"] == "HRI"], df[~df["Institute"].isin(["Alle", "HRI"])]])
    grafik_1[key] = df
    

grafik_1

{'unemp':     Variable  Year    Institute  Value  Prognose  \
 6        NaN  2026         Alle    NaN       6.2   
 0   UNEMPRBA  2026   <b>HRI</b>    6.5       NaN   
 1   UNEMPRBA  2026   DIW Berlin    6.2       NaN   
 2   UNEMPRBA  2026    IWH Halle    6.2       NaN   
 3   UNEMPRBA  2026     IfW Kiel    6.2       NaN   
 4   UNEMPRBA  2026    RWI Essen    6.2       NaN   
 5   UNEMPRBA  2026  ifo München    6.3       NaN   
 
                            einteilung  
 6            Alle Prognosen im Median  
 0  Das sagen die einzelnden Institute  
 1  Das sagen die einzelnden Institute  
 2  Das sagen die einzelnden Institute  
 3  Das sagen die einzelnden Institute  
 4  Das sagen die einzelnden Institute  
 5  Das sagen die einzelnden Institute  ,
 'cpi':   Variable  Year    Institute  Value  Prognose  \
 6      NaN  2026         Alle    NaN       2.1   
 0      CPI  2026   <b>HRI</b>    2.5       NaN   
 1      CPI  2026   DIW Berlin    2.1       NaN   
 2      CPI  2026    IWH 

In [319]:
#In Google Sheet laden
sheets = [(3, "unemp"), (4, "cpi"), (5, "gdp")]
for idx, key in sheets:
    ws = sh[idx]
    ws.clear()
    ws.set_dataframe(pd.DataFrame(grafik_1[key]), (1, 1))


In [320]:
#Auf Data Wrapper updaten
dw = Datawrapper(access_token="riHlVnwtZfCS3LPafNyE4QnO6PqbLSxuJhhs9yDEaI8Jq1UGUrb9WYhUWRx3H8ZK")

CHART_CPI = "iqMTw"
CHART_GDP = "7W9Gw"
CHART_UNEMP = "YzwfM" 

for chart_id in [CHART_CPI, CHART_GDP, CHART_UNEMP]: dw.publish_chart(chart_id)

/var/folders/01/8pcv5mtj69j_y6v97hckzr1r0000gn/T/ipykernel_8648/2845174760.py:8: DeprecationWarning: publish_chart() is deprecated and will be removed in a future version. Use the object-oriented chart classes instead. Example: chart = BarChart.get(chart_id='abc123'); chart.publish()
  for chart_id in [CHART_CPI, CHART_GDP, CHART_UNEMP]: dw.publish_chart(chart_id)


### Aussage 1 - Anteil der richtigen Prognosen

In [299]:
results = []
korridor = 0.1

# Loop durch die Metriken und Berechnungen der Abweichungen
for name, df in merged.items():
    
    # Abweichung zur Median-Prognose
    df['Abweichung'] = df['Value'] - df['Observed_Value']

    total = len(df['Abweichung'])
    pos = (df['Abweichung'] > korridor).sum() / total * 100
    neg = (df['Abweichung'] < -korridor).sum() / total * 100
    zero = 100 - pos - neg  # Rest den „keine Abweichung“ zuordnen

    # Ergebnisse speichern
    results.append({"Metrik": name,"Positiv": pos,"Negativ": neg,"Keine": zero})


abweichung = pd.DataFrame(results).set_index("Metrik")
abweichung



last10 = df[df["Year"] >= df["Year"].max() - 9].copy()

In [300]:
results = []
korridor = 0.1

# Loop durch die Metriken und Berechnungen der Abweichungen
for name, df in merged.items():

    df = df[df["Year"] >= df["Year"].max() - 9].copy()
    
    # Abweichung zur Median-Prognose
    df['Abweichung'] = df['Value'] - df['Observed_Value']

    total = len(df['Abweichung'])
    pos = (df['Abweichung'] > korridor).sum() / total * 100
    neg = (df['Abweichung'] < -korridor).sum() / total * 100
    zero = 100 - pos - neg  # Rest den „keine Abweichung“ zuordnen

    # Ergebnisse speichern
    results.append({"Metrik": name,"Positiv": pos,"Negativ": neg,"Keine": zero})


abweichung = pd.DataFrame(results).set_index("Metrik")
abweichung





,Positiv,Negativ,Keine
Metrik,,,
cpi,35.037433,43.662032,21.300535
gdp,66.950596,16.745165,16.304239
unemp,17.125492,44.905256,37.969253


### Grafik 2 und 3 - Die möglichen Abweichungen

In [301]:
# Prognosen in Quartale einteilen (analog zu Frühjahrs, Sommer, Herbst und Winterprognose)
quartale = {"Q1": [3, 4, 5], "Q2": [6, 7, 8],"Q3": [9, 10, 11],"Q4": [12, 1, 2]}

all_data_median = {}

for key, df in merged.items():
    df = df.copy()

    # Nur Beobachtungen mit schon vorhandenen wahren Werten benutzen
    df = df[df["Observed_Value"].notna()]

    # Prognosedatum in Prognosemonat und Jahr splitten
    df["Vintage"] = pd.to_datetime(df["Vintage"])
    df["Month"] = df["Vintage"].dt.month
    df["Year_Vintage"] = df["Vintage"].dt.year 

    # In Prognosequartal und Quartalsjahr aufteilen (wichtig, damit nicht z.B. die erste und letzte Prognose des gleichen Jahres in die gleiche Prognoseperiode gerechnet wird)
    df[["Quarter", "Quarter_Year"]] = df.apply(lambda row: pd.Series((next((q for q, months in quartale.items() if row["Month"] in months), None), row["Year_Vintage"] - 1 if row["Month"] in [1, 2] else row["Year_Vintage"])), axis=1)

    # Den jeweiligen Prognosemedian pro Jahr und Prognosezeitraum berechnen
    median_df = df.groupby(["Year", "Quarter", "Quarter_Year"]).agg({"Value": "median", "Observed_Value": "first"}).reset_index()

    all_data_median[key] = median_df

all_data_median



{'cpi':        Year Quarter  Quarter_Year  Value  Observed_Value
 0    1999.0      Q1          1998   1.80             1.2
 1    1999.0      Q1          1999   0.55             1.2
 2    1999.0      Q2          1998   1.40             1.2
 3    1999.0      Q2          1999   0.60             1.2
 4    1999.0      Q3          1998   1.35             1.2
 ..      ...     ...           ...    ...             ...
 280  2025.0      Q3          2024   2.10             2.2
 281  2025.0      Q3          2025   2.15             2.2
 282  2025.0      Q4          2023   2.25             2.2
 283  2025.0      Q4          2024   2.20             2.2
 284  2025.0      Q4          2025   2.20             2.2
 
 [285 rows x 5 columns],
 'gdp':        Year Quarter  Quarter_Year  Value  Observed_Value
 0    2000.0      Q1          1999    2.4             3.1
 1    2000.0      Q1          2000    2.8             3.1
 2    2000.0      Q2          1999    2.4             3.1
 3    2000.0      Q2          2

In [302]:
# Durchschnittliche Abweichung ermitteln

median_diff = {}

for key, df in all_data_median.items():
    df = df.copy()

    # Abweichung berechnen
    df["diff"] = df["Observed_Value"] - df["Value"]

    # In Abweichungsrichtung teilen
    df["ueberschaetzt"] = df["diff"].where(df["diff"] < 0)
    df["unterschaetzt"] = df["diff"].where(df["diff"] > 0)

    # Medianabweichung pro quartalsmäßige Medianprognose nach Abweichungsrichtung
    median_diff_df = df[["ueberschaetzt", "unterschaetzt"]].median().reset_index()
    median_diff_df.columns = ["Type", "Median_Diff"]

    median_diff[key] = median_diff_df

median_diff



{'cpi':             Type  Median_Diff
 0  ueberschaetzt         -0.3
 1  unterschaetzt          0.4,
 'gdp':             Type  Median_Diff
 0  ueberschaetzt       -0.800
 1  unterschaetzt        0.675,
 'unemp':             Type  Median_Diff
 0  ueberschaetzt         -0.4
 1  unterschaetzt          0.4}

In [303]:
# Abweichung mit Prognosen verrechnen

grafik_2 = {}

# Getrennte Szenarien für gdp sowie unemp und cpi berechnen 
for key in ["gdp"]:

    df = grafik_1[key].copy()

    # Datensatz mit dem Median aus der vorherigen Berechnung
    df_median = df[df["Institute"] == "Alle"].reset_index(drop=True)
    median_diff_df = median_diff[key].set_index("Type")

    df_median["Optimistisches Szenario"] = df_median["Prognose"] + median_diff_df.loc["unterschaetzt", "Median_Diff"]

    df_median["Pessimistisches Szenario"] = df_median["Prognose"] + median_diff_df.loc["ueberschaetzt", "Median_Diff"]

    # Säubern und umwandeln
    df_median["Pessimistisches Szenario"] = df_median["Pessimistisches Szenario"].astype(float).round(1).astype(str)
    df_median["Optimistisches Szenario"] = df_median["Optimistisches Szenario"].astype(float).round(1).astype(str)
    df_median.drop(columns=["Value", "Year", "einteilung", "Variable", "Institute"], inplace=True)

    grafik_2[key] = df_median

for key in ["cpi", "unemp"]:

    df = grafik_1[key].copy()

    # Datensatz mit dem Median aus der vorherigen Berechnung
    df_median = df[df["Institute"] == "Alle"].reset_index(drop=True)
    median_diff_df = median_diff[key].set_index("Type")

    df_median["Optimistisches Szenario"] = df_median["Prognose"] + median_diff_df.loc["ueberschaetzt", "Median_Diff"]

    df_median["Pessimistisches Szenario"] = df_median["Prognose"] + median_diff_df.loc["unterschaetzt", "Median_Diff"]

    # Säubern und umwandeln
    df_median["Pessimistisches Szenario"] = df_median["Pessimistisches Szenario"].astype(float).round(1).astype(str)
    df_median["Optimistisches Szenario"] = df_median["Optimistisches Szenario"].astype(float).round(1).astype(str)
    df_median.drop(columns=["Value", "Year", "einteilung", "Variable", "Institute"], inplace=True)

    grafik_2[key] = df_median




grafik_2



{'gdp':    Prognose Optimistisches Szenario Pessimistisches Szenario
 0       1.1                     1.8                      0.3,
 'cpi':    Prognose Optimistisches Szenario Pessimistisches Szenario
 0       2.1                     1.8                      2.5,
 'unemp':    Prognose Optimistisches Szenario Pessimistisches Szenario
 0       6.2                     5.8                      6.6}

In [304]:
#In Google Sheet laden
sheets = [(6, "unemp"), (7, "cpi"), (8, "gdp")]
for idx, key in sheets:
    ws = sh[idx]
    ws.clear()
    ws.set_dataframe(pd.DataFrame(grafik_2[key]), (1, 1))

In [305]:
#Auf Datawrapper updaten

CHART_CPI_p = "y1GUn"
CHART_GDP_p = "8r9dg"
CHART_UNEMP_p = "tLou6" 

CHART_CPI_n = "dlYTL"
CHART_GDP_n = "AxgeS"
CHART_UNEMP_n = "TQ3tr" 

for chart_id in [CHART_CPI_p, CHART_GDP_p, CHART_UNEMP_p, CHART_CPI_n, CHART_GDP_n, CHART_UNEMP_n]: dw.publish_chart(chart_id)

/var/folders/01/8pcv5mtj69j_y6v97hckzr1r0000gn/T/ipykernel_8648/843645784.py:11: DeprecationWarning: publish_chart() is deprecated and will be removed in a future version. Use the object-oriented chart classes instead. Example: chart = BarChart.get(chart_id='abc123'); chart.publish()
  for chart_id in [CHART_CPI_p, CHART_GDP_p, CHART_UNEMP_p, CHART_CPI_n, CHART_GDP_n, CHART_UNEMP_n]: dw.publish_chart(chart_id)


### Grafik 4 - Historisch

In [306]:
'''
grafik_4 = {}

for key, df in merged.items():
    df = df.copy()
    df = df[df["Observed_Value"].notna()]

    df["Vintage"] = pd.to_datetime(df["Vintage"])
    df["Month"] = df["Vintage"].dt.month
    df["Year_Vintage"] = df["Vintage"].dt.year 

    median_df = df.groupby(["Year"]).agg({"Value": "median", "Observed_Value": "first"}).reset_index()

    median_df["Value"] = median_df["Value"].astype(float).round(1).astype(str)
    median_df = median_df[median_df["Year"].between(2015, 2025)]
    median_df["Year"] = median_df["Year"].astype(float).astype(int).astype(str)
    
    grafik_4[key] = median_df

grafik_4
'''



'\ngrafik_4 = {}\n\nfor key, df in merged.items():\n    df = df.copy()\n    df = df[df["Observed_Value"].notna()]\n\n    df["Vintage"] = pd.to_datetime(df["Vintage"])\n    df["Month"] = df["Vintage"].dt.month\n    df["Year_Vintage"] = df["Vintage"].dt.year \n\n    median_df = df.groupby(["Year"]).agg({"Value": "median", "Observed_Value": "first"}).reset_index()\n\n    median_df["Value"] = median_df["Value"].astype(float).round(1).astype(str)\n    median_df = median_df[median_df["Year"].between(2015, 2025)]\n    median_df["Year"] = median_df["Year"].astype(float).astype(int).astype(str)\n\n    grafik_4[key] = median_df\n\ngrafik_4\n'

In [307]:
grafik_4 = {}

for key, df in all_data_median.items():
    df = df.copy()
    df = df[df["Observed_Value"].notna()]
    
    median_df = df.groupby(["Year"]).agg({"Value": "median", "Observed_Value": "first"}).reset_index()

    median_df["Medianprognose"] = median_df["Value"].astype(float).round(1).astype(str)
    median_df["Wahrer Wert"] = median_df["Observed_Value"].astype(float).round(1).astype(str)
    median_df = median_df[median_df["Year"].between(2015, 2025)]
    median_df["Year"] = median_df["Year"].astype(float).astype(int).astype(str)
    
    # Einen Wert bei gleichen Wert auf Missing setzten (schöner für die Visualisierung)
    #median_df.loc[median_df["Medianprognose"] == median_df["Wahrer Wert"],"Medianprognose"] = "NA"

    grafik_4[key] = median_df

grafik_4



{'cpi':     Year  Value  Observed_Value Medianprognose Wahrer Wert
 16  2015  0.500             0.3            0.5         0.3
 17  2016  0.500             1.7            0.5         1.7
 18  2017  1.700             1.7            1.7         1.7
 19  2018  1.800             1.7            1.8         1.7
 20  2019  1.500             1.5            1.5         1.5
 21  2020  0.775             1.0            0.8         1.0
 22  2021  1.825             3.1            1.8         3.1
 23  2022  2.725             7.9            2.7         7.9
 24  2023  6.000             5.9            6.0         5.9
 25  2024  2.500             2.2            2.5         2.2
 26  2025  2.125             2.2            2.1         2.2,
 'gdp':     Year  Value  Observed_Value Medianprognose Wahrer Wert
 15  2015   1.80             1.7            1.8         1.7
 16  2016   1.80             1.9            1.8         1.9
 17  2017   1.55             2.2            1.5         2.2
 18  2018   1.60         

In [308]:
#In Google Sheet laden
sheets = [(9, "unemp"), (10, "cpi"), (11, "gdp")]
for idx, key in sheets:
    ws = sh[idx]
    ws.clear()
    ws.set_dataframe(pd.DataFrame(grafik_4[key]), (1, 1))

In [309]:
#Auf Datawrapper updaten

CHART_CPI_all = "fDeue"
CHART_GDP_all = "WXyTi"
CHART_UNEMP_all = "QFngE" 

for chart_id in [CHART_CPI_all, CHART_GDP_all, CHART_UNEMP_all]: dw.publish_chart(chart_id)

/var/folders/01/8pcv5mtj69j_y6v97hckzr1r0000gn/T/ipykernel_8648/1295006387.py:7: DeprecationWarning: publish_chart() is deprecated and will be removed in a future version. Use the object-oriented chart classes instead. Example: chart = BarChart.get(chart_id='abc123'); chart.publish()
  for chart_id in [CHART_CPI_all, CHART_GDP_all, CHART_UNEMP_all]: dw.publish_chart(chart_id)


Grafik5 (Im Text als zweite Grafik)

In [310]:
grafik5 = {}

# Verschiedene KOrridore definieren
korridor01 = 0.1
korridor05 = 0.5
korridor1 = 1

for name, df in merged.items():
    
    # Nur die letzten 10 Jahre benutzen 
    last10 = df[df["Year"] >= df["Year"].max() - 9].copy()
    last10["Abweichung"] = last10["Value"] - last10["Observed_Value"]
    
    # Länge des Datensatzes definieren (für die Prozentrechnung)
    total = len(last10)

    # Absoluter Wert der Abweichung (also -1 wird zu |1|)
    abs_abw = last10["Abweichung"].abs()

    # Absolute Abweichungen in verschiedene Abweichungsgruppen einteilen
    gruppe01 = (abs_abw <= korridor01).sum() / total * 100
    gruppe05 = ((abs_abw > korridor01) & (abs_abw <= korridor05)).sum() / total * 100
    gruppe1 = ((abs_abw > korridor05) & (abs_abw <= korridor1)).sum() / total * 100
    gruppe11 = (abs_abw > korridor1).sum() / total * 100

    abw = {"Metrik": name, "weniger als 0,1 PP": gruppe01, "0,1–0,5 PP": gruppe05, "0,5–1 PP": gruppe1,"mehr als 1 PP": gruppe11}

    # Gerundeten Wert in dictionary speichern
    grafik5[name] = (pd.DataFrame([abw]).set_index("Metrik").round(1).astype(str))

grafik5

{'cpi':        weniger als 0,1 PP 0,1–0,5 PP 0,5–1 PP mehr als 1 PP
 Metrik                                                     
 cpi                  20.4       42.5     13.1          23.1,
 'gdp':        weniger als 0,1 PP 0,1–0,5 PP 0,5–1 PP mehr als 1 PP
 Metrik                                                     
 gdp                  15.4       28.6     15.6          39.6,
 'unemp':        weniger als 0,1 PP 0,1–0,5 PP 0,5–1 PP mehr als 1 PP
 Metrik                                                     
 unemp                37.2       36.5     19.4           6.1}

In [311]:
#In Google Sheet laden
sheets = [(12, "cpi"), (13, "unemp"), (14, "gdp")]
for idx, key in sheets:
    ws = sh[idx]
    ws.clear()
    ws.set_dataframe(pd.DataFrame(grafik5[key]), (1, 1))

In [312]:
#Auf Datawrapper updaten

CHART_CPI_abw = "ScJEf"
CHART_GDP_abw = "lraSy"
CHART_UNEMP_abw = "pnEI5" 

for chart_id in [CHART_CPI_abw, CHART_GDP_abw, CHART_UNEMP_abw]: dw.publish_chart(chart_id)

/var/folders/01/8pcv5mtj69j_y6v97hckzr1r0000gn/T/ipykernel_8648/1658108385.py:7: DeprecationWarning: publish_chart() is deprecated and will be removed in a future version. Use the object-oriented chart classes instead. Example: chart = BarChart.get(chart_id='abc123'); chart.publish()
  for chart_id in [CHART_CPI_abw, CHART_GDP_abw, CHART_UNEMP_abw]: dw.publish_chart(chart_id)
